<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Open AI API Connectors
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style = 'font-size:20px;font-family:Arial'><b>Introduction</b></p>
<p style = 'font-size:16px;font-family:Arial'>Teradata’s OpenAI API Connectivity functions enable seamless integration between Teradata and a wide range of Large Language Models (LLMs) using standard APIs such as chat completion and embeddings. This scalable capability allows organizations to connect to any LLM provider or custom solution, including OpenAI, Azure, Gemini, Mistral and even self-hosted models—without dependency on a single ecosystem.</p>
<p style = 'font-size:18px;font-family:Arial'><b>Business Value</b>
<ul style = 'font-size:16px;font-family:Arial'><li><b>Faster AI Adoption</b>: Leverage pre-trained LLMs without building models from scratch, accelerating time-to-value.</li>
<li><b>Flexibility & Interoperability</b>: Connect to multiple providers or custom LLM gateways, supporting complex enterprise architectures.</li>
<li><b>Scalable Performance</b>: Proven high-throughput processing (e.g., large-scale transcript analysis) with predictable performance and parallel scalability.</li>
<li><b>Cost & Efficiency Optimization</b>: AI MapReduce approach breaks large problems into smaller tasks, reducing token usage, improving response quality, and overcoming model limitations.</li>
<li><b>Advanced Analytics Enablement</b>: Supports use cases like summarization, classification, pattern detection, and insight generation directly within SQL workflows.</li></ul></p>
<p style = 'font-size:18px;font-family:Arial'><b>Why Teradata</b>
<ul style = 'font-size:16px;font-family:Arial'><li><b>Open Ecosystem (No Vendor Lock-in)</b>: Works with any LLM that supports standard APIs, unlike closed ecosystems.</li>
<li><b>Enterprise-Grade Integration</b>: Handles real-world complexity such as proxies, gateways, security layers, and custom configurations.</li>
<li><b>Built-In AI Processing in SQL</b>: Enables AI-driven analytics directly within Teradata using native functions and tools.</li>
<li><b>Performance at Scale</b>: High parallelism and efficient processing for large enterprise datasets.</li>
<li><b>Future-Ready Architecture</b>: Supports both hosted and self-hosted models, ensuring adaptability to evolving AI landscapes.</li></ul></p>
<p style = 'font-size:16px;font-family:Arial'>For more information you can check the following <a href = 'https://downloads.teradata.com/download/extensibility/open-ai-api-connectivity-functions'>link</a></p>

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>1. Initiate a connection to Teradata Database</b>

<p style = 'font-size:16px;font-family:Arial'>In the section, we import the required libraries and set environment variables and environment paths (if required).

In [ ]:
from teradataml import *

# Modify the following to match the specific client environment settings
display.max_rows = 5

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial'><b>1.1 Connect to Teradata Database</b></p>
<p style = 'font-size:16px;font-family:Arial'>You will be prompted to provide the password. Enter your password, press the Enter key, and then use the down arrow to go to the next cell.</p>

In [ ]:
%run -i ../../UseCases/startup.ipynb
eng = create_context(host = 'host.docker.internal', username='dbc', password = password)
print(eng)

In [ ]:
%%capture
execute_sql('''SET query_band='DEMO=Open_AI_API_Connectors_Python.ipynb;' UPDATE FOR SESSION; ''')

<p style = 'font-size:16px;font-family:Arial'>Begin running steps with Shift + Enter keys. </p>

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>1.2 Setting up the Database and Functions for the connectors</b></p>
<p style = 'font-size:16px;font-family:Arial'>In this section we will be running the commands to create the database and functions needed for the connector to run. For more information you can check the following <a href = 'https://downloads.teradata.com/download/extensibility/open-ai-api-connectivity-functions'>link</a></p>

In [ ]:
with open("commands.json", "r") as file:
    data = json.load(file)

for item in data["queries"]:
    try:
        execute_sql(item["query"])
    except Exception as e:
        print(f"The initialization steps have already been executed for this environment!")            
        #print(f"Error: {e}")
        pass

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>1.3 Reconnecting with demo_user</b></p>


In [ ]:
%run -i ../../UseCases/startup.ipynb
eng = create_context(host = 'host.docker.internal', username='demo_user', password = password)
print(eng)

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>1.4 Getting Data for This Demo</b></p>

<p style = 'font-size:16px;font-family:Arial'>We have provided data for this demo on cloud storage. You can either run the demo using foreign tables to access the data without any storage on your environment or download the data to local storage, which may yield faster execution. Still, there could be considerations of available storage. Two statements are in the following cell, and one is commented out. You may switch which mode you choose by changing the comment string.</p>

In [ ]:
# %run -i ../../UseCases/run_procedure.py "call get_data('DEMO_Email_cloud');"        # Takes 30seconds
%run -i ../../UseCases/run_procedure.py "call get_data('DEMO_Email_local');"        # Takes 1 minute

<p style = 'font-size:16px;font-family:Arial'>Next is an optional step – if you want to see the status of databases/tables created and space used.</p>

In [ ]:
%run -i ../../UseCases/run_procedure.py "call space_report();"        # Takes 10 seconds

<p style = 'font-size:16px;font-family:Arial'>Now our setup is complete, we can use the connector for connecting LLMs 

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>2. ChatCompletion Table Operator</b>
<p style = 'font-size:16px;font-family:Arial'>In this section we will see how to use ChatCompletion Table Operator to call OpenAI-compatible chat completion APIs directly from Teradata SQL queries.<br></p>

<div class="alert alert-block alert-info">
<p style = 'font-size:16px;font-family:Arial'><b>Note: </b><i>This notebook is using OpenAI models hence OpenAI key will be needed </i></p>
</div>

<p style = 'font-size:16px;font-family:Arial'>Let's examine the sample email data we'll be processing. Each row contains an id and a email_text column with email content that we'll analyze using the LLM.

In [ ]:
d1=DataFrame(in_schema("demo_email","email_data"))
d1

In [ ]:
# Securely input the OpenAI API key (will not be displayed)
api_key = getpass.getpass("Enter your OpenAI API key: ")

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.1 Simple Text Summarization</b></p>
<p style = 'font-size:16px;font-family:Arial'>The most basic use case - ask the LLM to summarize each email in 10 words or less. The <b>CompleteChat </b>function processes each row independently and adds the response in a new <b>response_txt</b> column.

In [ ]:
# Basic summarization - each email is summarized in 10 words or less
DataFrame.from_query(f'''
    SELECT * 
    FROM openai_client.CompleteChat(
        ON (SELECT id, email_text as txt FROM demo_email.email_data sample 10)
        USING
            BaseURL('https://api.openai.com')
            SystemMessage('You are the text summarizer. Give me the summary of the given text. No longer than 10 words')
            Model('gpt-4o-mini')
            ApiKey('{api_key}')
    ) a;
''')

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.2 Structured JSON Output</b></p>
<p style = 'font-size:16px;font-family:Arial'>For more advanced use cases, you can instruct the LLM to return structured JSON data. This is useful for:
<ul style = 'font-size:16px;font-family:Arial'>
    <li>Entity extraction</li>
    <li>Classification tasks</li>
    <li>Data enrichment pipelines</li>
</ul>
<p style = 'font-size:16px;font-family:Arial'>
    The <b>BodyParameters</b> option enables <b>response_format:{"type":"json_object"}</b> to ensure valid JSON output. We then use Teradata's JSON functions to extract individual fields.

In [ ]:
# Extract structured data from emails using JSON output format
# The response is parsed as JSON and individual fields are extracted as columns
DataFrame.from_query(f'''
    SELECT
        id,
        txt,
        CAST(response_txt AS JSON).main_subject AS main_subject,
        CAST(response_txt AS JSON).secondary_subject AS secondary_subject
    FROM openai_client.CompleteChat(
        ON (SELECT id, email_text as txt FROM demo_email.email_data sample 10)
        USING
            BaseURL('https://api.openai.com')
            SystemMessage('You are the text analyzer. Give me two main subjects of the email.
                           Respond in JSON format: {{"main_subject": "<main subject>", "secondary_subject": "<secondary subject>"}}.
                           You must output valid JSON only, with no markdown, no backticks, and no code fences.')
            Model('gpt-4o-mini')
            ApiKey('{api_key}')
            BodyParameters('response_format:{{"type":"json_object"}}')
    ) a;
''')

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>2.3 Rate Limiting and Retry Handling</b></p>
<p style = 'font-size:16px;font-family:Arial'>When processing large datasets, you may encounter API rate limits (HTTP 429). The Table Operator provides built-in retry handling with configurable delays.
<ul style = 'font-size:16px;font-family:Arial'><b>Key parameters:</b>
<li><b>Delays('200,400,800')</b> - Retry delays in milliseconds (escalating)</li>
<li><b>RetriesNumber(5)</b> - Maximum number of retry attempts </li>
<li><b>OutputProcessingDetails('True')</b> - Adds diagnostic columns to monitor performance  </li>
<li><b>ThrowErrorOnRateLimit('False')</b> - Continue processing even if rate limited (returns NULL instead of error) </li>
</ul>
<ul style = 'font-size:16px;font-family:Arial'><b>Diagnostic columns added:</b>
<li> <b>retries_made</b> - Number of retries performed                             </li>
<li> <b>last_attempt_duration</b> - Time taken for the last API call (ms)          </li>
<li> <b>rate_limit_exceeded</b> - 1 if rate limited after all retries, 0 otherwise </li>
<li> <b>rate_limit_exceeded_error_details</b> - Error message if rate limited      </li>
</ul>


In [ ]:
# Production-ready query with retry handling and diagnostics
DataFrame.from_query(f'''
    SELECT * 
    FROM openai_client.CompleteChat(
        ON (SELECT id, email_text as txt FROM demo_email.email_data sample 10)
        USING
            BaseURL('https://api.openai.com')
            SystemMessage('You are the text summarizer. Give me the summary of the given text. No longer than 10 words')
            Model('gpt-4o-mini')
            ApiKey('{api_key}')
            
            -- Rate limiting configuration
            Delays('200,400,800')           -- Escalating retry delays in ms
            RetriesNumber(5)                -- Max retry attempts
            OutputProcessingDetails('True') -- Add diagnostic columns
            ThrowErrorOnRateLimit('False')  -- Don't fail on rate limit, return NULL instead
    ) a;
''')

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>3. Embeddings Table Operator</b>
<p style = 'font-size:16px;font-family:Arial'>In this section we will see how to use the **Embeddings** Table Operator to generate text embeddings and perform semantic search using Teradata's vector capabilities.</p>

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>3.1 Generate Embeddings (COLUMNAR Format)</b></p>
<p style = 'font-size:16px;font-family:Arial'>Generate embeddings for all documents. The COLUMNAR format creates one column per embedding dimension, which is required for TD_VectorDistance.
<ul style = 'font-size:16px;font-family:Arial'><b>Key parameters:</b>
<li> <b>EmbeddingLength(1536)</b> - Must match the model's output dimension                                      </li>
<li> <b>EmbeddingPrefix('emb_')</b> - Column name prefix (creates emb_0, emb_1, ..., emb_1535)</li>
<li> <b>OutputFormat('COLUMNAR')</b> - One FLOAT column per dimension (default)</li>
</ul>


In [ ]:
execute_sql(f'''
    CREATE TABLE embedding_store AS (
        SELECT * 
        FROM openai_client.Embeddings(
            ON (SELECT id, email_text as txt FROM demo_email.email_data)
            USING
                BaseURL('https://api.openai.com')
                Model('text-embedding-3-small')
                ApiKey('{api_key}')
                EmbeddingLength(1536)
                OutputFormat('COLUMNAR')
                EmbeddingPrefix('emb_')
        ) a
    ) WITH DATA ;
''')

print("Embeddings generated and stored in 'embedding_store' table.")

In [ ]:
emb_data=DataFrame("embedding_store")
emb_data

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>3.2 Create Search Query Embedding</b></p>
<p style = 'font-size:16px;font-family:Arial'>To perform semantic search, we need to convert our search query into an embedding using the same model.

In [ ]:
# Define search query and generate its embedding
search_query = "renewable energy and sustainability projects"

execute_sql(f'''
    CREATE TABLE query_embedding AS (
        SELECT * 
        FROM openai_client.Embeddings(
            ON (SELECT 1 AS id, '{search_query}' AS txt)
            USING
                BaseURL('https://api.openai.com')
                Model('text-embedding-3-small')
                ApiKey('{api_key}')
                EmbeddingLength(1536)
                OutputFormat('COLUMNAR')
                EmbeddingPrefix('emb_')
        ) a
    ) WITH DATA;
''')

print(f"Query embedding generated for: '{search_query}'")

In [ ]:
qry_data=DataFrame("query_embedding")
qry_data

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial'><b>3.3 Semantic Search with TD_VectorDistance</b></p>
<p style = 'font-size:16px;font-family:Arial'>Now we perform semantic search using cosine similarity. The TD_VectorDistance function calculates the distance between the query embedding and all document embeddings in our store.
<ul style = 'font-size:16px;font-family:Arial'><b>How it works:</b>
<li> <b>TD_VectorDistance</b> - computes cosine similarity between vectors </li>
<li> Lower distance = higher similarity (for cosine distance)</li>
<li> Results are ordered by distance to find the most relevant documents</li>
 </ul>
<p style = 'font-size:16px;font-family:Arial'>This leverages Teradata's Massive Parallel Processing (MPP) for high-performance, scalable computation.

In [ ]:
DataFrame.from_query(f"""
SELECT 
    dt.target_id, 
    dt.reference_id,
    e_ref.email_text as reference_txt,
    (1.0 - dt.distance) as similiarity 
FROM
    TD_VECTORDISTANCE (
        ON query_embedding AS TargetTable
        ON embedding_store AS ReferenceTable DIMENSION
        USING
            TargetIDColumn('id')
            TargetFeatureColumns('[emb_0:emb_1535]')
            RefIDColumn('id')
            RefFeatureColumns('[emb_0:emb_1535]')
            DistanceMeasure('cosine')
            topk(3)
    ) AS dt
JOIN demo_email.email_data e_ref on e_ref.id = dt.reference_id;
""")

<ul style = 'font-size:16px;font-family:Arial'>The results are ordered by cosine distance (ascending):
    <li>Lower distance = More semantically similar to the query</li>
    <li>Higher distance = Less similar</li>
</ul>
<p style = 'font-size:16px;font-family:Arial'>
The top results should be documents that discuss topics related to our search query ("renewable energy and sustainability projects"), even if they don't contain the exact keywords.
</p>

<hr style="height:2px;border:none;">
<b style = 'font-size:20px;font-family:Arial'>4. Cleanup</b>

<p style = 'font-size:18px;font-family:Arial'><b>Work Tables</b></p>
<p style = 'font-size:16px;font-family:Arial'>The following code will clean up intermediate tables.</p>

In [ ]:
db_drop_table(table_name='query_embedding')

In [ ]:
db_drop_table(table_name='embedding_store')

<p style = 'font-size:18px;font-family:Arial'><b>Databases and Tables</b></p>
<p style = 'font-size:16px;font-family:Arial'>The following code will clean up tables and databases created above.</p>

In [ ]:
%run -i ../../UseCases/run_procedure.py "call remove_data('DEMO_Email');" 
#Takes 45 seconds

In [ ]:
remove_context()

<hr style="height:1px;border:none;">

<p style = 'font-size:16px;font-family:Arial'><b>Links:</b></p>
<ul style = 'font-size:16px;font-family:Arial'>
    <li>Open AI API connectors on Teradata downloads: <a href = 'https://downloads.teradata.com/download/extensibility/open-ai-api-connectivity-functions'>here</a></li>
    </ul>

<footer style="padding-bottom:35px; border-bottom:3px solid #91A0Ab">
      <div style="float:right;">
        <div style="float:left; margin-top:14px">
            Copyright © Teradata - 2026. All Rights Reserved
        </div>
    </div>
</footer>